In [2]:
import pandas as pd
import numpy as np

# 表 1：1 月销售数据 (规范命名)
jan_sales = pd.DataFrame({
    'order_id': [101, 102, 103],
    'p_id': ['A', 'B', 'C'],
    'qty': [2, 1, 5]
})

# 表 2：2 月销售数据 (非规范命名 + 重复项)
# 陷阱：列名大小写不同、存在重复录入、存在 1 月没出现过的新产品 D
feb_sales = pd.DataFrame({
    'Order_ID': [201, 202, 202, 203],
    'P_ID': ['B', 'C', 'C', 'D'],
    'QTY': [1, 3, 3, 10]
})

# 表 3：产品信息维度表 (主键重复陷阱)
# 陷阱：产品 A 有两条记录（可能是历史价格），会导致 Join 爆炸
products = pd.DataFrame({
    'p_id': ['A', 'A', 'B', 'C'],
    'price': [10.5, 11.0, 20.0, 100.0]
})

In [ ]:
# convert the column names to  lowercase
jan_sales.columns = [col.strip().lower() for col in jan_sales.columns]
feb_sales.columns = [col.strip().lower() for col in feb_sales.columns]

# check if the column names are consistent
print(jan_sales.columns.equals(feb_sales.columns))

# vitical merger
concat_df = pd.concat([jan_sales,feb_sales],ignore_index=True)


# remove duplicates
df_all = concat_df.drop_duplicates(subset='order_id',keep='last',ignore_index=True)
print(df_all)

# associate df_all and products
df_final = pd.merge(df_all,products,on='p_id',how='left',validate='many_to_many',indicator=True)
print(df_final)

# caculate the difference in the number of rows
diff = len(df_final) - len(df_all)
if diff > 0:
    print(f"⚠️ 警告：检测到数据膨胀！合并后增加了 {diff} 行。")  
# locate the source of the explosion
duplicated_products = products[products['p_id'].duplicated(keep=False)]
print("--- products 表中的重复定义（爆炸源） ---")
print(duplicated_products)

# capture  "Orphan data"
no_price = df_final[df_final['_merge'] == 'left_only']
print(no_price)

True
   order_id p_id  qty
0       101    A    2
1       102    B    1
2       103    C    5
3       201    B    1
4       202    C    3
5       203    D   10
   order_id p_id  qty  price     _merge
0       101    A    2   10.5       both
1       101    A    2   11.0       both
2       102    B    1   20.0       both
3       103    C    5  100.0       both
4       201    B    1   20.0       both
5       202    C    3  100.0       both
6       203    D   10    NaN  left_only
⚠️ 警告：检测到数据膨胀！合并后增加了 1 行。
--- products 表中的重复定义（爆炸源） ---
  p_id  price
0    A   10.5
1    A   11.0
